In [1]:
import os
import json
import re  
import torch
import librosa
import numpy as np
import time
from tqdm import tqdm
from scipy.signal import find_peaks
from transformers import Wav2Vec2Processor, Wav2Vec2Model 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # التوزيع الآمن
from sklearn.svm import SVC 
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
from scipy.signal import find_peaks

try:
    import parselmouth
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "praat-parselmouth"])
    import parselmouth
    
try:
    from xgboost import XGBClassifier
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

#print(f"⏳ جاري تحميل وتأمين الموديل العربي في الذاكرة على جهاز ({device})...")
print(f"⏳ Loading and allocating the Arabic model in memory on ({device})...")
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)
model.eval()
#print("✅ تم تحميل الموديل اللغوي بنجاح!\n")
print("✅ Language model loaded successfully!\n")

def extract_hnr_praat(file_path):
    try:
        snd = parselmouth.Sound(file_path)
        harmonicity = snd.to_harmonicity_cc(
            time_step=0.01,
            minimum_pitch=100,
            silence_threshold=0.1,
            periods_per_window=4.5
        )

        values = harmonicity.values.flatten()
        values = values[np.isfinite(values)]
        values = values[values > -200]  # إزالة القيم غير الصالحة من Praat

        if len(values) == 0:
            return 16.0

        return float(np.mean(values))

    except Exception:
        return 16.0
    
def extract_hybrid_features_raw(file_path):
    try:
        speech, sr = librosa.load(file_path, sr=16000)
        inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)
        input_values = inputs.input_values.to(device)
        
        with torch.no_grad():
            outputs = model(input_values)
            deep_embeddings = torch.mean(outputs.last_hidden_state, dim=1).squeeze().cpu().numpy()
        
        # القياسات الفسيولوجية
        f0, _, _ = librosa.pyin(speech, fmin=100, fmax=500, sr=16000)
        f0_clean = f0[~np.isnan(f0)]
        local_jitter = 0.015 if len(f0_clean) < 2 else np.mean(np.abs(np.diff(f0_clean))) / np.mean(f0_clean)
            
        peaks, _ = find_peaks(speech, distance=int(16000/300))
        amplitudes = np.abs(speech[peaks])
        amplitudes_clean = amplitudes[amplitudes > 0.01]
        local_shimmer = 0.035 if len(amplitudes_clean) < 2 else np.mean(np.abs(np.diff(amplitudes_clean))) / np.mean(amplitudes_clean)
            
       #n = len(speech)
        #        r = np.correlate(speech, speech, mode='full')[n-1:]
      #  low_lag, high_lag = int(16000 / 500), int(16000 / 100)
     #   max_r = np.max(r[low_lag:high_lag]) if len(r) > high_lag else 0
    #    total_energy = r[0] if len(r) > 0 else 1
     #   hnr = 20.0 if total_energy - max_r <= 0 else 10 * np.log10(max_r / (total_energy - max_r))
        

        hnr = extract_hnr_praat(file_path)

        local_jitter = 0.015 if np.isnan(local_jitter) else local_jitter
        local_shimmer = 0.035 if np.isnan(local_shimmer) else local_shimmer
        hnr = 16.0 if np.isnan(hnr) or np.isinf(hnr) else hnr
        
        return deep_embeddings, np.array([local_jitter, local_shimmer, hnr])
    except Exception:
        return None, None


def process_clinical_dataset_robust(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    records_list = data.get('records', [])
        
    all_deep_features = []
    all_clinical_features = []
    all_labels = []
    
    folder_name = "Audio" if os.path.exists("Audio") else "audio"
    #print(f"📂 جاري فحص ومسح محتويات مجلد الصوت الحقيقي: '{folder_name}'...")
    print(f"📂 Scanning and checking the contents of the real audio folder: '{folder_name}'...")
    
    audio_file_registry = {}
    if os.path.exists(folder_name):
        for f_name in os.listdir(folder_name):
            numbers = re.findall(r'\d+', f_name)
            if numbers: audio_file_registry[numbers[0]] = os.path.join(folder_name, f_name)
                
    #print(f"✅ تم بنجاح فهرسة {len(audio_file_registry)} ملف صوتي حقيقي.")
    print(f"✅ {len(audio_file_registry)} real audio files have been successfully indexed.")
    #print(f"🚀 البدء الفعلي في استخلاص الميزات لـ {len(records_list)} سجل...")
    print(f"🚀 Starting feature extraction for {len(records_list)} records...")
    
    start_time = time.time()
    for item in tqdm(records_list, desc="📊  feature extraction"):
        audio_path = item.get('audio_path', '')
        actual_path = None
        if audio_path:
            json_numbers = re.findall(r'\d+', os.path.basename(audio_path))
            if json_numbers: actual_path = audio_file_registry.get(json_numbers[0])
        
        if actual_path and os.path.exists(actual_path):
            deep_feat, clinic_feat = extract_hybrid_features_raw(actual_path)
            if deep_feat is not None:
                all_deep_features.append(deep_feat)
                all_clinical_features.append(clinic_feat)
                all_labels.append(1 if item.get('error', False) else 0)
                
    #print(f"\n⏱️ انتهت عملية المعالجة العميقة في: {(time.time() - start_time)/60:.2f} دقيقة.")
    print(f"\n⏱️ Processing finished in: {(time.time() - start_time)/60:.2f} mins.")
    
    X_deep_np = np.array(all_deep_features)
    X_clinical_np = np.array(all_clinical_features)
    y_np = np.array(all_labels)
    
    #print(f"📊 إجمالي السجلات المستخلصة بنجاح: {X_deep_np.shape[0]}")
    print(f"📊 Total extracted records: {X_deep_np.shape[0]}")
        
    splits_np = np.array([item.get('split', '') for item in records_list])

    idx_train = np.where(splits_np == "train")[0]
    idx_test  = np.where(splits_np == "test")[0]
    idx_val   = np.where(splits_np == "val")[0]


    overlap = np.intersect1d(idx_train, idx_test)

    print(f"📌 Official training size: {len(idx_train)}")
    print(f"📌 Official test size: {len(idx_test)}")
    print(f"📌 Train/Test overlap: {len(overlap)}")

    return (X_deep_np[idx_train], X_clinical_np[idx_train], y_np[idx_train],
            X_deep_np[idx_val],   X_clinical_np[idx_val],   y_np[idx_val],
            X_deep_np[idx_test], X_clinical_np[idx_test], y_np[idx_test])

json_file_path = "arabic_child_speech_mispronunciation_with_phoneme_alignment.json"
X_train_deep, X_train_clinical, y_train,X_val_deep, X_val_clinical, y_val, X_test_deep, X_test_clinical, y_test = process_clinical_dataset_robust(json_file_path)

#print("\n📊 حجم مصفوفات الأطروحة النهائية والمحسنة بعد التقسيم الموزون:")
print("\n📊 Final optimized thesis matrix shapes after weighted split:")
#print(f"📈 ميزات التدريب العميقة الفعليّة: {X_train_deep.shape}")
print(f"📈 Deep train features shape: {X_train_deep.shape}")
#print(f"📉 ميزات الاختبار العميقة الفعليّة: {X_test_deep.shape}")
print(f"📈 Deep test features shape: {X_test_deep.shape}")

X_train_raw = np.hstack((X_train_deep, X_train_clinical))
X_test_raw = np.hstack((X_test_deep, X_test_clinical))
X_val_raw = np.hstack((X_val_deep, X_val_clinical))


scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_raw)
X_val_scaled   = scaler_final.transform(X_val_raw)
X_test_scaled = scaler_final.transform(X_test_raw)

#print("\n⏳ جاري تحليل وتدريب XGBoost لاستخلاص أفضل 100 ميزة ذهبية...")
print("\n⏳ Running XGBoost to extract top 100 golden features...")
estimated_ratio = np.sum(y_train == 0) / np.sum(y_train == 1)

xgb_advanced = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=estimated_ratio,
    eval_metric='logloss', random_state=42
)
xgb_advanced.fit(X_train_scaled, y_train)

importances = xgb_advanced.feature_importances_
top_k_indices = np.argsort(importances)[::-1][:100]


X_train_selected = X_train_scaled[:, top_k_indices]
X_val_selected   = X_val_scaled[:, top_k_indices]
X_test_selected  = X_test_scaled[:, top_k_indices]

#print("⏳ جاري تدريب الـ SVM المطور على المصفوفة المصفاة الحيوية...")
print("⏳ Training advanced SVM on the filtered matrix...")
svm_advanced = SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced', probability=True, random_state=42)
svm_advanced.fit(X_train_selected, y_train)

def find_best_threshold(model, X, y):
    probs = model.predict_proba(X)[:, 1]
    best_th, max_acc = 0.5, 0
    for th in np.arange(0.3, 0.7, 0.01):
        acc = accuracy_score(y, (probs >= th).astype(int))
        if acc > max_acc: max_acc, best_th = acc, th
    return best_th, max_acc

best_th_xgb, max_val_acc_xgb = find_best_threshold(xgb_advanced, X_val_scaled, y_val)
best_th_svm, max_val_acc_svm = find_best_threshold(svm_advanced, X_val_selected, y_val)

y_pred_svm_test = (
    svm_advanced.predict_proba(X_test_selected)[:, 1] >= best_th_svm
).astype(int)

print("Final Test Accuracy:", accuracy_score(y_test, y_pred_svm_test) * 100)

y_pred_xgb_opt = (xgb_advanced.predict_proba(X_test_scaled)[:, 1] >= best_th_xgb).astype(int)
y_pred_svm_opt = (svm_advanced.predict_proba(X_test_selected)[:, 1] >= best_th_svm).astype(int)


# Probabilities for AUC
y_prob_xgb = xgb_advanced.predict_proba(X_test_scaled)[:, 1]
y_prob_svm = svm_advanced.predict_proba(X_test_selected)[:, 1]

# AUC-ROC
auc_xgb = roc_auc_score(y_test, y_prob_xgb)
auc_svm = roc_auc_score(y_test, y_prob_svm)

# Final predictions using threshold
y_pred_xgb_opt = (y_prob_xgb >= best_th_xgb).astype(int)
y_pred_svm_opt = (y_prob_svm >= best_th_svm).astype(int)

print(f"AUC-ROC (Hybrid XGBoost): {auc_xgb:.4f}")
print(f"AUC-ROC (Hybrid SVM): {auc_svm:.4f}")

print("Final XGBoost Test Accuracy:", accuracy_score(y_test, y_pred_xgb_opt) * 100)
print("Final SVM Test Accuracy:", accuracy_score(y_test, y_pred_svm_opt) * 100)



# --- XGBoost Evaluation ---
print("\n📋 XGBoost Advanced Classification Report:")
print(classification_report(y_test, y_pred_xgb_opt, target_names=['Normal (0)', 'Disordered (1)']))

print("\n🛑 New XGBoost Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_opt))
print("="*75)


#print("\n📋 تقرير التصنيف الطبي المطور لـ SVM:")
print("\n📋 SVM Advanced Medical Classification Report:")
#print(classification_report(y_test, y_pred_svm_opt, target_names=['سليم (0)', 'مضطرب (1)']))
print(classification_report(y_test, y_pred_svm_opt, target_names=['Normal (0)', 'Disordered (1)']))
#print("\n🛑 مصفوفة الارتباك الجديدة لـ SVM:\n", confusion_matrix(y_test, y_pred_svm_opt))
print("\n🛑 New SVM Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm_opt))
print("="*75)

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading and allocating the Arabic model in memory on (cpu)...


Loading weights: 100%|██████████| 422/422 [00:00<00:00, 16013.72it/s]
Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Language model loaded successfully!

📂 Scanning and checking the contents of the real audio folder: 'Audio'...
✅ 2000 real audio files have been successfully indexed.
🚀 Starting feature extraction for 2000 records...


📊  feature extraction: 100%|██████████| 2000/2000 [22:01<00:00,  1.51it/s]



⏱️ Processing finished in: 22.02 mins.
📊 Total extracted records: 2000
📌 Official training size: 1375
📌 Official test size: 378
📌 Train/Test overlap: 0

📊 Final optimized thesis matrix shapes after weighted split:
📈 Deep train features shape: (1375, 1024)
📈 Deep test features shape: (378, 1024)

⏳ Running XGBoost to extract top 100 golden features...
⏳ Training advanced SVM on the filtered matrix...
Final Test Accuracy: 64.81481481481481
AUC-ROC (Hybrid XGBoost): 0.7138
AUC-ROC (Hybrid SVM): 0.7460
Final XGBoost Test Accuracy: 59.25925925925925
Final SVM Test Accuracy: 64.81481481481481

📋 XGBoost Advanced Classification Report:
                precision    recall  f1-score   support

    Normal (0)       0.62      0.74      0.68       219
Disordered (1)       0.52      0.38      0.44       159

      accuracy                           0.59       378
     macro avg       0.57      0.56      0.56       378
  weighted avg       0.58      0.59      0.58       378


🛑 New XGBoost Confusio

In [2]:
import pandas as pd

pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred_svm_opt
}).to_csv("Hybrid_SVM_predictions.csv", index=False)

print("✅ Hybrid_SVM_predictions.csv saved")

✅ Hybrid_SVM_predictions.csv saved


In [3]:
import pandas as pd

pd.DataFrame({
    "y_true": y_test,
    "y_pred": y_pred_xgb_opt
}).to_csv("Hybrid_XGBoost_predictions.csv", index=False)

print("✅ Hybrid_XGBoost_predictions.csv saved")

✅ Hybrid_XGBoost_predictions.csv saved


In [4]:
!pip install statsmodels -q

import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

def run_mcnemar(file_a, file_b, name_a, name_b):
    df_a = pd.read_csv(file_a)
    df_b = pd.read_csv(file_b)

    y_true_a = df_a["y_true"].values
    y_true_b = df_b["y_true"].values

    if not np.array_equal(y_true_a, y_true_b):
        raise ValueError("❌ y_true غير متطابق بين الملفين. لازم نفس ترتيب عينات test.")

    pred_a = df_a["y_pred"].values
    pred_b = df_b["y_pred"].values
    y_true = y_true_a

    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # table:
    # [[both wrong, A wrong B correct],
    #  [A correct B wrong, both correct]]
    table = np.zeros((2, 2), dtype=int)

    for ca, cb in zip(correct_a, correct_b):
        table[int(ca), int(cb)] += 1

    result = mcnemar(table, exact=False, correction=True)

    print("\n" + "="*80)
    print(f"McNemar Test: {name_a} vs {name_b}")
    print("="*80)
    print("Contingency Table:")
    print(table)
    print(f"Statistic: {result.statistic:.4f}")
    print(f"p-value  : {result.pvalue:.6f}")

    if result.pvalue < 0.05:
        print("✅ Significant difference (p < 0.05)")
    else:
        print("❌ No significant difference (p >= 0.05)")

# المقارنات الثلاثة
run_mcnemar(
    "FineTuning_predictions.csv",
    "Hybrid_SVM_predictions.csv",
    "Fine-tuned Wav2Vec2",
    "Hybrid SVM"
)

run_mcnemar(
    "FineTuning_predictions.csv",
    "Hybrid_XGBoost_predictions.csv",
    "Fine-tuned Wav2Vec2",
    "Hybrid XGBoost"
)

run_mcnemar(
    "Hybrid_SVM_predictions.csv",
    "Hybrid_XGBoost_predictions.csv",
    "Hybrid SVM",
    "Hybrid XGBoost"
)


McNemar Test: Fine-tuned Wav2Vec2 vs Hybrid SVM
Contingency Table:
[[ 33   6]
 [100 239]]
Statistic: 81.5943
p-value  : 0.000000
✅ Significant difference (p < 0.05)

McNemar Test: Fine-tuned Wav2Vec2 vs Hybrid XGBoost
Contingency Table:
[[ 32   7]
 [122 217]]
Statistic: 100.7442
p-value  : 0.000000
✅ Significant difference (p < 0.05)

McNemar Test: Hybrid SVM vs Hybrid XGBoost
Contingency Table:
[[113  20]
 [ 41 204]]
Statistic: 6.5574
p-value  : 0.010445
✅ Significant difference (p < 0.05)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.svm import SVC

def evaluate_model(name, y_true, y_prob, threshold=0.50):
    y_pred = (y_prob >= threshold).astype(int)

    print("\n" + "="*80)
    print(name)
    print("="*80)
    print(f"Threshold : {threshold:.2f}")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"Precision : {precision_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f"F1-score  : {f1_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f"AUC-ROC   : {roc_auc_score(y_true, y_prob):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=['Normal (0)', 'Disordered (1)']))

    return {
        "Model": name,
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred)*100,
        "Precision": precision_score(y_true, y_pred, zero_division=0)*100,
        "Recall": recall_score(y_true, y_pred, zero_division=0)*100,
        "F1": f1_score(y_true, y_pred, zero_division=0)*100,
        "AUC": roc_auc_score(y_true, y_prob)
    }


# ============================================================
# 1) Hybrid الكامل: XGBoost Feature Selection + Optimized Threshold
# ============================================================

y_prob_full = svm_advanced.predict_proba(X_test_selected)[:, 1]

res_full = evaluate_model(
    "1) Full Hybrid SVM: XGBoost Feature Selection + Optimized Threshold",
    y_test,
    y_prob_full,
    best_th_svm
)


# ============================================================
# 2) Hybrid SVM بدون Threshold Optimization
# نفس الموديل ونفس XGBoost-selected features، لكن threshold = 0.50
# ============================================================

res_no_threshold = evaluate_model(
    "2) Hybrid SVM without Threshold Optimization",
    y_test,
    y_prob_full,
    0.50
)


# ============================================================
# 3) Hybrid SVM بدون XGBoost Feature Selection
# تدريب SVM على كل الميزات: Wav2Vec2 embeddings + Jitter + Shimmer + HNR
# واختيار threshold على Validation فقط
# ============================================================

svm_no_feature_selection = SVC(
    kernel='rbf',
    C=10.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm_no_feature_selection.fit(X_train_scaled, y_train)

best_th_no_fs, max_val_acc_no_fs = find_best_threshold(
    svm_no_feature_selection,
    X_val_scaled,
    y_val
)

y_prob_no_fs = svm_no_feature_selection.predict_proba(X_test_scaled)[:, 1]

res_no_fs = evaluate_model(
    "3) Hybrid SVM without XGBoost Feature Selection",
    y_test,
    y_prob_no_fs,
    best_th_no_fs
)


# ============================================================
# Summary Table
# ============================================================

import pandas as pd

ablation_results = pd.DataFrame([
    res_full,
    res_no_threshold,
    res_no_fs
])

print("\n" + "="*80)
print("Ablation Study Summary")
print("="*80)
print(ablation_results.round(4))


1) Full Hybrid SVM: XGBoost Feature Selection + Optimized Threshold
Threshold : 0.61
Accuracy  : 64.81%
Precision : 63.00%
Recall    : 39.62%
F1-score  : 48.65%
AUC-ROC   : 0.7460
Confusion Matrix:
[[182  37]
 [ 96  63]]
                precision    recall  f1-score   support

    Normal (0)       0.65      0.83      0.73       219
Disordered (1)       0.63      0.40      0.49       159

      accuracy                           0.65       378
     macro avg       0.64      0.61      0.61       378
  weighted avg       0.64      0.65      0.63       378


2) Hybrid SVM without Threshold Optimization
Threshold : 0.50
Accuracy  : 64.29%
Precision : 58.70%
Recall    : 50.94%
F1-score  : 54.55%
AUC-ROC   : 0.7460
Confusion Matrix:
[[162  57]
 [ 78  81]]
                precision    recall  f1-score   support

    Normal (0)       0.68      0.74      0.71       219
Disordered (1)       0.59      0.51      0.55       159

      accuracy                           0.64       378
     macro avg

In [5]:
print("\n" + "="*90)
print("FINAL MODEL COMPARISON")
print("="*90)

print(f"{'Model':45s} {'Threshold':>12s}")
print("-"*90)

print(f"{'Hybrid XGBoost':45s} {best_th_xgb:12.2f}")
print(f"{'Hybrid SVM (Full)':45s} {best_th_svm:12.2f}")
print(f"{'Hybrid SVM (Without Threshold Optimization)':45s} {'0.50':>12s}")
print(f"{'Hybrid SVM (Without XGBoost Feature Selection)':45s} {best_th_no_fs:12.2f}")

print("="*90)


FINAL MODEL COMPARISON
Model                                            Threshold
------------------------------------------------------------------------------------------
Hybrid XGBoost                                        0.59
Hybrid SVM (Full)                                     0.61
Hybrid SVM (Without Threshold Optimization)           0.50
Hybrid SVM (Without XGBoost Feature Selection)         0.53


In [6]:
import pandas as pd
from sklearn.metrics import accuracy_score

results = []

probs = svm_advanced.predict_proba(X_val_selected)[:, 1]

for th in np.arange(0.30, 0.70, 0.01):
    pred = (probs >= th).astype(int)
    acc = accuracy_score(y_val, pred)

    results.append([round(th, 2), acc * 100])

df = pd.DataFrame(results, columns=["Threshold", "Validation Accuracy (%)"])

print(df.sort_values("Validation Accuracy (%)", ascending=False).head(10))

    Threshold  Validation Accuracy (%)
31       0.61                73.684211
32       0.62                73.684211
36       0.66                72.874494
35       0.65                72.874494
33       0.63                72.874494
30       0.60                72.874494
34       0.64                72.874494
24       0.54                72.469636
29       0.59                72.469636
39       0.69                72.469636


In [8]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import numpy as np

# Frozen Wav2Vec2 embeddings فقط بدون Jitter/Shimmer/HNR
scaler_deep = StandardScaler()

X_train_deep_scaled = scaler_deep.fit_transform(X_train_deep)
X_val_deep_scaled   = scaler_deep.transform(X_val_deep)
X_test_deep_scaled  = scaler_deep.transform(X_test_deep)

svm_frozen_wav2vec2 = SVC(
    kernel='rbf',
    C=10.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

svm_frozen_wav2vec2.fit(X_train_deep_scaled, y_train)

# اختيار threshold من validation
def find_best_threshold(model, X, y):
    probs = model.predict_proba(X)[:, 1]
    best_th, max_acc = 0.5, 0
    for th in np.arange(0.30, 0.70, 0.01):
        pred = (probs >= th).astype(int)
        acc = accuracy_score(y, pred)
        if acc > max_acc:
            max_acc = acc
            best_th = th
    return best_th, max_acc

best_th_frozen, val_acc_frozen = find_best_threshold(
    svm_frozen_wav2vec2,
    X_val_deep_scaled,
    y_val
)

# التقييم النهائي على test
y_prob_frozen = svm_frozen_wav2vec2.predict_proba(X_test_deep_scaled)[:, 1]
y_pred_frozen = (y_prob_frozen >= best_th_frozen).astype(int)

print("="*80)
print("1) Frozen Wav2Vec2 without Fine-tuning")
print("="*80)
print(f"Selected Threshold : {best_th_frozen:.2f}")
print(f"Validation Accuracy: {val_acc_frozen*100:.2f}%")
print(f"Test Accuracy      : {accuracy_score(y_test, y_pred_frozen)*100:.2f}%")
print(f"Precision          : {precision_score(y_test, y_pred_frozen, zero_division=0)*100:.2f}%")
print(f"Recall             : {recall_score(y_test, y_pred_frozen, zero_division=0)*100:.2f}%")
print(f"F1-score           : {f1_score(y_test, y_pred_frozen, zero_division=0)*100:.2f}%")
print(f"AUC-ROC            : {roc_auc_score(y_test, y_prob_frozen):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_frozen))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_frozen, target_names=['Normal (0)', 'Disordered (1)']))

1) Frozen Wav2Vec2 without Fine-tuning
Selected Threshold : 0.53
Validation Accuracy: 78.14%
Test Accuracy      : 66.93%
Precision          : 63.08%
Recall             : 51.57%
F1-score           : 56.75%
AUC-ROC            : 0.7578

Confusion Matrix:
[[171  48]
 [ 77  82]]

Classification Report:
                precision    recall  f1-score   support

    Normal (0)       0.69      0.78      0.73       219
Disordered (1)       0.63      0.52      0.57       159

      accuracy                           0.67       378
     macro avg       0.66      0.65      0.65       378
  weighted avg       0.66      0.67      0.66       378



In [2]:
from transformers import Wav2Vec2ForCTC
import os
import re
import torch
import librosa
import numpy as np

print("⏳ جاري تحميل موديل المحاذاة الحرفية Wav2Vec2ForCTC...")
model_ctc = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(device)
model_ctc.eval()
print("✅ تم تحميل الموديل بنجاح!\n")

def diagnose_entire_folder(folder_path):
    if not os.path.exists(folder_path): 
        print(f"🛑 المجلد {folder_path} غير موجود!")
        return
    audio_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.wav')]
    
    print("\n" + "="*90)
    print(f"🔬 [نظام الفحص الشامل والمستمر - الجيل الثاني المطور المحمي]")
    print(f"📊 عدد الحالات المكتشفة للفحص الفوري: {len(audio_files)} طفل")
    print("="*90 + "\n")

    total_scanned = 0
    disorder_count_xgb = 0
    disorder_count_svm = 0

    arabic_diacritics = re.compile(r'[\u064B-\u0652]')

    for index, file_name in enumerate(audio_files, 1):
        file_path = os.path.join(folder_path, file_name)
        print(f"📁 [{index}/{len(audio_files)}] الملف الحالي: {file_name}\n" + "-"*80)
        
        deep_feat, clinic_feat = extract_hybrid_features_raw(file_path)
        if deep_feat is None: 
            print("⚠️ فشل استخراج الميزات الصوتية من الملف.")
            continue
        total_scanned += 1

        speech_signal, _ = librosa.load(file_path, sr=16000)
        inputs = processor(speech_signal, sampling_rate=16000, return_tensors="pt", padding=True)
        with torch.no_grad():
            logits = model_ctc(inputs.input_values.to(device)).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = processor.batch_decode(predicted_ids)[0].strip()
        
        target_word = os.path.splitext(file_name)[0].strip()
        t_clean = re.sub(r'\s+', '', target_word)
        o_clean = re.sub(r'\s+', '', transcription) if transcription else ""
        
        t_clean_pure = re.sub(arabic_diacritics, '', t_clean)
        o_clean_pure = re.sub(arabic_diacritics, '', o_clean)

        diff_pairs = []
        max_len = max(len(t_clean_pure), len(o_clean_pure))
        for i in range(max_len):
            char_t = t_clean_pure[i] if i < len(t_clean_pure) else "∅"
            char_o = o_clean_pure[i] if i < len(o_clean_pure) else "∅"
            if char_t != char_o:
                diff_pairs.append(f"{char_t} -> {char_o}")
        
        if len(t_clean_pure) == 0:
            cer_val = 0.0 if len(o_clean_pure) == 0 else 100.0
        elif t_clean_pure == o_clean_pure:
            cer_val = 0.0
        else:
            mismatches = sum(1 for t, o in zip(t_clean_pure, o_clean_pure) if t != o)
            len_diff = abs(len(t_clean_pure) - len(o_clean_pure))
            total_errors = mismatches + len_diff
            cer_val = min(100.0, (total_errors / len(t_clean_pure)) * 100)

        print(f"   📝 [التحليل اللغوي والفونيمي]:\n      ∟ الكلمة المستهدفة طبيّاً     : {target_word}\n      ∟ النص المنطوق من الطفل     : {transcription}\n      ∟ 📊 تحليل الفارق الحرفي    : {', '.join(diff_pairs) if diff_pairs else 'تطابق كامل'} (CER: {cer_val:.1f}%)")
        print(f"\n   🎙️ [القياسات الصوتية العيادية الحية]:\n      ∟ Jitter: {clinic_feat[0]*100:.3f}% | Shimmer: {clinic_feat[1]*100:.3f}% | HNR: {clinic_feat[2]:.2f} dB")
                
        single_raw = np.hstack((deep_feat.reshape(1, -1), clinic_feat.reshape(1, -1)))
        single_scaled = scaler_final.transform(single_raw)
                
        prob_xgb = xgb_advanced.predict_proba(single_scaled)[0, 1]
        pred_xgb = 1 if prob_xgb >= best_th_xgb else 0
                
        single_selected = single_scaled[:, top_k_indices]
        prob_svm = svm_advanced.predict_proba(single_selected)[0, 1]
        pred_svm = 1 if prob_svm >= best_th_svm else 0
        
        if pred_xgb == 1: disorder_count_xgb += 1
        if pred_svm == 1: disorder_count_svm += 1
        
        print(f"\n   🤖 [قرارات النمذجة الذكية المحدثة]:")
        print(f"      👈 تشخيص [XGBoost] -> {'🚨 مضطرب' if pred_xgb==1 else '✅ سليم'} (الاحتمالية: {prob_xgb*100:.1f}%)")
        print(f"      👈 تشخيص [SVM RBF]  -> {'🚨 مضطرب' if pred_svm==1 else '✅ سليم'} (الاحتمالية: {prob_svm*100:.1f}%)")
        print("-" * 80 + "\n")

        print(f"\n   ⚖️ التوصية السريرية والتشخيصية:")
        if pred_xgb == 1 and pred_svm == 1:
            print("      ∟ 🚨 [إحالة عاجلة للمختص]: يكتشف النظام علامات اضطراب نطق نوعي حاد في الفونيمات والنبرة.")
            print("        يوصى بضرورة عرض الطفل على أخصائي أمراض النطق والتخاطب للفحص السريري الدقيق.")
        elif pred_xgb == 1 or pred_svm == 1:
            print("      ∟ ⚠️ [متابعة عيادية حذرة]: يوجد تباين إحصائي بين النماذج الذكية في تحديد الإصابة.")
            print("        يُنصح بإعادة الفحص الاستكشافي أو استشارة الأخصائي لضمان دقة المراقبة الطبية.")
        else:
            print("      ∟ ✅ [حالة مستقرة]: النطق فسيولوجياً وفونيمياً سليم وضمن النطاق التلوّري الطبيعي المستقر ولا حاجة لمختص.")
            
        print("-" * 85 + "\n")

    print("="*90 + f"\n📊 [التقرير الختامي الشامل للمجلد الجديد]\n"+"="*90)
    print(f"🔹 إجمالي الملفات المفحوصة بنجاح    : {total_scanned}")
    print(f"🔹 الحالات المصابة حسب نظام XGBoost : {disorder_count_xgb} طفل")
    print(f"🔹 الحالات المصابة حسب نظام SVM المطور : {disorder_count_svm} طفل")
    print("="*90 + "\n")

diagnose_entire_folder("Test_Child")

⏳ جاري تحميل موديل المحاذاة الحرفية Wav2Vec2ForCTC...


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 35806.17it/s]


✅ تم تحميل الموديل بنجاح!


🔬 [نظام الفحص الشامل والمستمر - الجيل الثاني المطور المحمي]
📊 عدد الحالات المكتشفة للفحص الفوري: 5 طفل

📁 [1/5] الملف الحالي: حصان.wav
--------------------------------------------------------------------------------
   📝 [التحليل اللغوي والفونيمي]:
      ∟ الكلمة المستهدفة طبيّاً     : حصان
      ∟ النص المنطوق من الطفل     : حصان
      ∟ 📊 تحليل الفارق الحرفي    : تطابق كامل (CER: 0.0%)

   🎙️ [القياسات الصوتية العيادية الحية]:
      ∟ Jitter: 5.042% | Shimmer: 34.076% | HNR: 4.83 dB

   🤖 [قرارات النمذجة الذكية المحدثة]:
      👈 تشخيص [XGBoost] -> ✅ سليم (الاحتمالية: 16.9%)
      👈 تشخيص [SVM RBF]  -> ✅ سليم (الاحتمالية: 10.2%)
--------------------------------------------------------------------------------


   ⚖️ التوصية السريرية والتشخيصية:
      ∟ ✅ [حالة مستقرة]: النطق فسيولوجياً وفونيمياً سليم وضمن النطاق التلوّري الطبيعي المستقر ولا حاجة لمختص.
-------------------------------------------------------------------------------------

📁 [2/5] الملف الحال

In [7]:
import gradio as gr
import re
import os
import pandas as pd
import numpy as np
import torch
import librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2ForSequenceClassification

if 'model_ctc' not in locals():
    print("⏳ Loading Phonetic Alignment Model (Wav2Vec2ForCTC)...")
    model_ctc = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME).to(device)
    model_ctc.eval()
    print("✅ CTC Model loaded successfully!")

if 'model_sequence' not in locals():
    print("⏳ Loading Pure Deep Learning Classifier (Wav2Vec2ForSequenceClassification)...")
    model_sequence = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    model_sequence.eval()
    print("✅ Sequence Classification Model loaded successfully!")

# Clean Arabic diacritics and calculate Character Error Rate (CER) Safely
def calculate_safe_cer(t_clean, o_clean):
    if not t_clean:
        return 0.0, "Target word not provided"
    
    arabic_diacritics = re.compile(r'[\u064B-\u0652]')
    t_clean = re.sub(arabic_diacritics, '', t_clean)
    o_clean = re.sub(arabic_diacritics, '', o_clean)
    
    if t_clean == o_clean:
        return 0.0, "100% Full Match"
        
    diff_pairs = []
    max_len = max(len(t_clean), len(o_clean))
    for i in range(max_len):
        char_t = t_clean[i] if i < len(t_clean) else "∅"
        char_o = o_clean[i] if i < len(o_clean) else "∅"
        if char_t != char_o:
            diff_pairs.append(f"[{char_t} ➔ {char_o}]")
            
    mismatches = sum(1 for t, o in zip(t_clean, o_clean) if t != o)
    len_diff = abs(len(t_clean) - len(o_clean))
    total_errors = mismatches + len_diff
    
    cer_val = min(100.0, (total_errors / len(t_clean)) * 100)
    diff_analysis = ", ".join(diff_pairs) if diff_pairs else "Full Match"
    
    return cer_val, diff_analysis

# 1. Batch Folder Processing Function (Test_Child)
def diagnose_folder_gui(folder_path="Test_Child"):
    if not os.path.exists(folder_path): 
        return f"🛑 Folder '{folder_path}' not found in the current working directory!", None
        
    audio_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.wav')]
    if not audio_files:
        return f"⚠️ Folder '{folder_path}' is empty or contains no WAV files.", None
        
    total_scanned = 0
    disorder_count_xgb = 0
    disorder_count_svm = 0
    disorder_count_wav = 0
    detailed_rows = []
    
    best_th_wav = 0.50

    for file_name in audio_files:
        file_path = os.path.join(folder_path, file_name)
        deep_feat, clinic_feat = extract_hybrid_features_raw(file_path)
        if deep_feat is None: continue
        
        total_scanned += 1

        # Transcription using Wav2Vec2 CTC
        speech_signal, _ = librosa.load(file_path, sr=16000)
        inputs = processor(speech_signal, sampling_rate=16000, return_tensors="pt", padding=True)
        inputs_device = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            logits_ctc = model_ctc(inputs_device['input_values']).logits
        predicted_ids = torch.argmax(logits_ctc, dim=-1)
        transcription = processor.batch_decode(predicted_ids)[0].strip()
        
        target_word = os.path.splitext(file_name)[0].strip()
        cer_val, _ = calculate_safe_cer(re.sub(r'\s+', '', target_word), re.sub(r'\s+', '', transcription) if transcription else "")


        with torch.no_grad():
            outputs_seq = model_sequence(inputs_device['input_values'])
            logits_seq = outputs_seq.logits
            probs_seq = torch.softmax(logits_seq, dim=-1).cpu().numpy()[0]
            prob_wav_disorder = probs_seq[1]
            pred_wav = 1 if prob_wav_disorder >= best_th_wav else 0
        
        single_raw = np.hstack((deep_feat.reshape(1, -1), clinic_feat.reshape(1, -1)))
        single_scaled = scaler_final.transform(single_raw)
        
        prob_xgb = xgb_advanced.predict_proba(single_scaled)[0, 1]
        pred_xgb = 1 if prob_xgb >= best_th_xgb else 0
        
        single_selected = single_scaled[:, top_k_indices]
        prob_svm = svm_advanced.predict_proba(single_selected)[0, 1]
        pred_svm = 1 if prob_svm >= best_th_svm else 0

        #print("XGBoost Features:", xgb_advanced.n_features_in_)
        #print("SVM Features:", svm_advanced.n_features_in_)
        
        if pred_xgb == 1: disorder_count_xgb += 1
        if pred_svm == 1: disorder_count_svm += 1
        if pred_wav == 1: disorder_count_wav += 1
                
        final_status = "🚨 Disorder Detected" if (pred_xgb == 1 and pred_svm == 1) else ("⚠️ Clinical Monitoring" if (pred_xgb == 1 or pred_svm == 1) else "✅ Normal Speech")
        
        detailed_rows.append({
            "File Name": file_name,
            "Target Word": target_word,
            "Child Speech": transcription if transcription else "---",
            "CER (Normalized)": f"{cer_val:.1f}%",
            "Jitter": f"{clinic_feat[0]*100:.3f}%",
            "Shimmer": f"{clinic_feat[1]*100:.3f}%",
            "HNR (dB)": f"{clinic_feat[2]:.2f}", 
            "XGBoost Prediction": f"{'Disorder' if pred_xgb==1 else 'Normal'} ({prob_xgb*100:.1f}%)",
            "SVM Prediction": f"{'Disorder' if pred_svm==1 else 'Normal'} ({prob_svm*100:.1f}%)",
             "Final Status": final_status,
            "Wav2Vec2 Deep Classifier without Fine_Tuning": f"{'Disorder' if pred_wav==1 else 'Normal'} ({prob_wav_disorder*100:.1f}%)"           
        })
    
    summary_report = (
        f"📊 [Batch Screening Summary Statistical Report]\n"
        f"==================================================\n"
        f"• Total successfully scanned records : {total_scanned} children\n"
        f"• Speech Disorder cases detected by XGBoost  : {disorder_count_xgb}\n"
        f"• Speech Disorder cases detected by SVM      : {disorder_count_svm}\n"
        f"• Speech Disorder cases detected by Wav2Vec2 : {disorder_count_wav}\n"
        f"==================================================\n"
        f"⚙️ [Dynamic AI Optimization Thresholds Injected]:\n"
        f" ∟ Optimal Decision Boundary (XGBoost)  : {best_th_xgb:.2f}\n"
        f" ∟ Optimal Decision Boundary (SVM RBF)  : {best_th_svm:.2f}\n"
        f" ∟ Optimal Decision Boundary (Wav2Vec2) : {best_th_wav:.2f}"
    )
    
    df_results = pd.DataFrame(detailed_rows)
    return summary_report, df_results


def diagnose_single_gui(audio_file_path, target_word):
    if audio_file_path is None:
        return "⚠️ Please upload or record an audio file first.", "", "", ""
    
    deep_feat, clinic_feat = extract_hybrid_features_raw(audio_file_path)
    if deep_feat is None: 
        return "⚠️ Failed to extract acoustic clinical features from this file.", "", "", ""

    speech_signal, _ = librosa.load(audio_file_path, sr=16000)
    inputs = processor(speech_signal, sampling_rate=16000, return_tensors="pt", padding=True)
    inputs_device = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        logits_ctc = model_ctc(inputs_device['input_values']).logits
        outputs_seq = model_sequence(inputs_device['input_values'])
    
    predicted_ids = torch.argmax(logits_ctc, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0].strip()
        
    best_th_wav = 0.50
    probs_seq = torch.softmax(outputs_seq.logits, dim=-1).cpu().numpy()[0]
    prob_wav_disorder = probs_seq[1]
    pred_wav = 1 if prob_wav_disorder >= best_th_wav else 0
    status_seq = "🚨 Speech Disorder Detected" if pred_wav == 1 else "✅ Normal Speech Pattern"

    cer_val, diff_analysis = calculate_safe_cer(re.sub(r'\s+', '', target_word) if target_word else "", re.sub(r'\s+', '', transcription) if transcription else "")

    # Hybrid Classifiers Phase
    single_raw = np.hstack((deep_feat.reshape(1, -1), clinic_feat.reshape(1, -1)))
    single_scaled = scaler_final.transform(single_raw)
    
    prob_xgb = xgb_advanced.predict_proba(single_scaled)[0, 1]
    pred_xgb = 1 if prob_xgb >= best_th_xgb else 0
    
    single_selected = single_scaled[:, top_k_indices]
    prob_svm = svm_advanced.predict_proba(single_selected)[0, 1]
    pred_svm = 1 if prob_svm >= best_th_svm else 0
    
    status_xgb = "🚨 Speech Disorder Detected" if pred_xgb == 1 else "✅ Normal Speech Pattern"
    status_svm = "🚨 Speech Disorder Detected" if pred_svm == 1 else "✅ Normal Speech Pattern"
    
    report_models = (
        f"🤖 [Hybrid] XGBoost Model Classifier: {status_xgb} ({prob_xgb*100:.1f}%) [Threshold: {best_th_xgb:.2f}]\n"
        f"🤖 [Hybrid] SVM (RBF Kernel) Classifier: {status_svm} ({prob_svm*100:.1f}%) [Threshold: {best_th_svm:.2f}]\n"
        f"🧠 [Pure Deep] Wav2Vec2 Sequence Classifier: {status_seq} ({prob_wav_disorder*100:.1f}%) [Threshold: {best_th_wav:.2f}]"
    )
    
    report_clinical = (
        f"• Jitter (Frequency Instability Index)   : {clinic_feat[0]*100:.3f}%\n"
        f"• Shimmer (Amplitude Micro-Fluctuation) : {clinic_feat[1]*100:.3f}%\n"
        f"• Harmonic-to-Noise Ratio (HNR Glottal) : {clinic_feat[2]:.2f} dB"
    )
    
    report_linguistic = (
        f"• Target Word : {target_word if target_word else 'Not Specified'}\n"
        f"• Recognized  : {transcription if transcription else '[Silence]'}\n"
        f"• CER         : {cer_val:.1f}%"
    )
    
    recommendation = "✅ Architectural evaluation successfully executed across all three optimized models."
    return report_models, report_clinical, report_linguistic, recommendation


# --- Gradio ---
with gr.Blocks(theme=gr.themes.Soft(), title="Intelligent Speech Assessment System") as demo:
    gr.Markdown("# 🔬 Automatic Mispronunciation Detection in Arabic-Speaking Children Using a Hybrid Wav2Vec2 and Acoustic-Feature Framework")
    with gr.Tabs():
        with gr.TabItem("📁 Batch Folder Diagnostics (Test_Child)"):
            folder_input = gr.Textbox(value="Test_Child", label="Target Folder Path")
            folder_btn = gr.Button("🚀 Run High-Throughput Screening", variant="primary")
            out_folder_summary = gr.Textbox(label="📊 Statistical Summary, Classifier Outputs & Optimal Thresholds", interactive=False, lines=10)
            out_folder_table = gr.Dataframe(label="📋 Patient Analytics Matrix (Extracts Metrics, Bio-Acoustics & Three Classifier Logs)")
            folder_btn.click(fn=diagnose_folder_gui, inputs=[folder_input], outputs=[out_folder_summary, out_folder_table])
            
        with gr.TabItem("🎙️ Individual Case Diagnostics"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio Input")
                    word_input = gr.Textbox(label="Target Word")
                    submit_btn = gr.Button("🔍 Run Real-Time Diagnostics", variant="primary")
                with gr.Column():
                    out_models = gr.Textbox(label="🤖 Classifiers, Probabilities & Boundaries", interactive=False, lines=4)
                    out_clinical = gr.Textbox(label="🎙️ Bio-Acoustic Markers", interactive=False, lines=4)
                    out_linguistic = gr.Textbox(label="📝 Computational Linguistics", interactive=False, lines=4)
                    out_recommendation = gr.Textbox(label="⚖️ Final Recommendation", interactive=False, lines=2)
            submit_btn.click(fn=diagnose_single_gui, inputs=[audio_input, word_input], outputs=[out_models, out_clinical, out_linguistic, out_recommendation])

demo.launch(inline=True, share=False)

⏳ Loading Phonetic Alignment Model (Wav2Vec2ForCTC)...


Loading weights: 100%|██████████| 424/424 [00:00<00:00, 21263.64it/s]


✅ CTC Model loaded successfully!
⏳ Loading Pure Deep Learning Classifier (Wav2Vec2ForSequenceClassification)...


Loading weights: 100%|██████████| 422/422 [00:00<00:00, 19941.60it/s]
Wav2Vec2ForSequenceClassification LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-arabic
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
projector.bias    | MISSING    | 
classifier.bias   | MISSING    | 
projector.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\HP\AppData\Local\Temp\ipykernel_4116\760801016.py:211: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Intelligent Speech Assessment Syst

✅ Sequence Classification Model loaded successfully!
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
# ============================================================
# Baselines: MFCC + SVM and eGeMAPS + SVM
# Same split as current Hybrid code: 80/20, random_state=42
# ============================================================

import os, json, re, subprocess, sys
import numpy as np
import librosa
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,roc_auc_score
)

try:
    import opensmile
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "opensmile"])
    import opensmile

json_file_path = "arabic_child_speech_mispronunciation_with_phoneme_alignment.json"


def print_metrics(name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print("\n" + "="*70)
    print(f"📊 {name}")
    print("="*70)
    print(f"Accuracy    : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print(f"Specificity : {specificity*100:.2f}%")
    print(f"F1-score    : {f1_score(y_true, y_pred, zero_division=0)*100:.2f}%")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Disordered"]))
    print("="*70)


def find_best_threshold_f1(model, X, y):
    probs = model.predict_proba(X)[:, 1]
    best_th, best_f1 = 0.5, 0

    for th in np.arange(0.10, 0.90, 0.01):
        pred = (probs >= th).astype(int)
        f1 = f1_score(y, pred, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    return best_th, best_f1


def extract_mfcc_features(file_path, sr=16000, n_mfcc=40):
    speech, _ = librosa.load(file_path, sr=sr)

    mfcc = librosa.feature.mfcc(y=speech, sr=sr, n_mfcc=n_mfcc)

    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std  = np.std(mfcc, axis=1)

    return np.concatenate([mfcc_mean, mfcc_std])


smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

def extract_egemaps_features(file_path):
    features = smile.process_file(file_path)
    return features.values.flatten()


with open(json_file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

records = data["records"]

folder_name = "Audio" if os.path.exists("Audio") else "audio"

audio_file_registry = {}
for fname in os.listdir(folder_name):
    nums = re.findall(r"\d+", fname)
    if nums:
        audio_file_registry[nums[0]] = os.path.join(folder_name, fname)

print(f"✅ Indexed audio files: {len(audio_file_registry)}")


X_mfcc = []
X_egemaps = []
y_all = []
splits_all = []

skipped = 0

for item in tqdm(records, desc="Extracting MFCC + eGeMAPS"):
    audio_path = item.get("audio_path", "")
    nums = re.findall(r"\d+", os.path.basename(audio_path))

    if not nums:
        skipped += 1
        continue

    actual_path = audio_file_registry.get(nums[0])

    if not actual_path or not os.path.exists(actual_path):
        skipped += 1
        continue

    try:
        mfcc_feat = extract_mfcc_features(actual_path)
        egemaps_feat = extract_egemaps_features(actual_path)

        X_mfcc.append(mfcc_feat)
        X_egemaps.append(egemaps_feat)
        y_all.append(1 if item.get("error", False) else 0)
        splits_all.append(item.get("split", ""))
    except Exception:
        skipped += 1

X_mfcc = np.array(X_mfcc)
X_egemaps = np.array(X_egemaps)
y_all = np.array(y_all)
splits_all = np.array(splits_all)
print("Total extracted:", len(y_all))
print("Skipped:", skipped)
print("MFCC shape:", X_mfcc.shape)
print("eGeMAPS shape:", X_egemaps.shape)


idx_train = np.where(splits_all == "train")[0]
idx_val   = np.where(splits_all == "val")[0]
idx_test  = np.where(splits_all == "test")[0]

overlap = np.intersect1d(idx_train, idx_test)

print("Official train size:", len(idx_train))
print("Official val size:", len(idx_val))
print("Official test size:", len(idx_test))
print("Train/Test overlap:", len(overlap))


# ============================================================
# MFCC + SVM
# ============================================================
scaler_mfcc = StandardScaler()
X_train_mfcc = scaler_mfcc.fit_transform(X_mfcc[idx_train])
X_test_mfcc  = scaler_mfcc.transform(X_mfcc[idx_test])

svm_mfcc = SVC(
    kernel="rbf",
    C=10.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm_mfcc.fit(X_train_mfcc, y_all[idx_train])
X_val_mfcc = scaler_mfcc.transform(X_mfcc[idx_val])

best_th_mfcc, val_f1_mfcc = find_best_threshold_f1(
    svm_mfcc,
    X_val_mfcc,
    y_all[idx_val]
)

# Probabilities
y_prob_mfcc = svm_mfcc.predict_proba(X_test_mfcc)[:, 1]

# AUC
auc_mfcc = roc_auc_score(y_all[idx_test], y_prob_mfcc)

# Final prediction using threshold
y_pred_mfcc = (y_prob_mfcc >= best_th_mfcc).astype(int)

print(f"AUC-ROC : {auc_mfcc:.4f}")

print(f"\nBest threshold for MFCC + SVM: {best_th_mfcc:.2f}")
print_metrics("MFCC + SVM baseline on official test split", y_all[idx_test], y_pred_mfcc)


# ============================================================
# eGeMAPS + SVM
# ============================================================
scaler_egemaps = StandardScaler()
X_train_egemaps = scaler_egemaps.fit_transform(X_egemaps[idx_train])
X_test_egemaps  = scaler_egemaps.transform(X_egemaps[idx_test])

svm_egemaps = SVC(
    kernel="rbf",
    C=10.0,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)

svm_egemaps.fit(X_train_egemaps, y_all[idx_train])
X_val_egemaps = scaler_egemaps.transform(X_egemaps[idx_val])

best_th_egemaps, val_f1_egemaps = find_best_threshold_f1(
    svm_egemaps,
    X_val_egemaps,
    y_all[idx_val]
)

# Probabilities
y_prob_egemaps = svm_egemaps.predict_proba(X_test_egemaps)[:, 1]

# AUC
auc_egemaps = roc_auc_score(y_all[idx_test], y_prob_egemaps)

# Final prediction
y_pred_egemaps = (y_prob_egemaps >= best_th_egemaps).astype(int)

print(f"AUC-ROC : {auc_egemaps:.4f}")

print(f"\nBest threshold for eGeMAPS + SVM: {best_th_egemaps:.2f}")
print_metrics("eGeMAPS + SVM baseline on official test split", y_all[idx_test], y_pred_egemaps)

✅ Indexed audio files: 2000


Extracting MFCC + eGeMAPS: 100%|██████████| 2000/2000 [01:18<00:00, 25.60it/s]


Total extracted: 2000
Skipped: 0
MFCC shape: (2000, 80)
eGeMAPS shape: (2000, 88)
Official train size: 1375
Official val size: 247
Official test size: 378
Train/Test overlap: 0
AUC-ROC : 0.4629

Best threshold for MFCC + SVM: 0.25

📊 MFCC + SVM baseline on official test split
Accuracy    : 42.06%
Precision   : 26.92%
Recall      : 22.01%
Specificity : 56.62%
F1-score    : 24.22%

Confusion Matrix:
[[124  95]
 [124  35]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.50      0.57      0.53       219
  Disordered       0.27      0.22      0.24       159

    accuracy                           0.42       378
   macro avg       0.38      0.39      0.39       378
weighted avg       0.40      0.42      0.41       378

AUC-ROC : 0.6753

Best threshold for eGeMAPS + SVM: 0.10

📊 eGeMAPS + SVM baseline on official test split
Accuracy    : 57.67%
Precision   : 49.82%
Recall      : 86.16%
Specificity : 36.99%
F1-score    : 63.13%

Confusion Mat